ollama list
ollama run qwen3.5:9b
ollama pull nomic-embed-text
pip install llama-index-core llama-index-readers-file llama-index-llms-ollama llama-index-embeddings-ollama llama-index-vector-stores-chroma chromadb

In [14]:
print(chroma_collection.count())
result = chroma_collection.peek(limit=3)

print(result)

11
{'ids': ['f6f18b36-e598-425e-9900-9a222b302cd1', 'cb2c2f4a-f3ee-4778-9462-25220636ccb5', 'dd4ef6c3-cf80-496c-9e7f-b1b4758e8a3c'], 'embeddings': array([[ 0.03977739,  0.03177289, -0.17200816, ..., -0.05524087,
        -0.06301346, -0.02325391],
       [ 0.0801122 ,  0.04366473, -0.15709339, ..., -0.0446601 ,
        -0.08072832, -0.01410257],
       [ 0.07643017,  0.04767303, -0.16647647, ..., -0.03932884,
        -0.08547321, -0.02051553]], shape=(3, 768)), 'documents': ["INDUSTRIAL INSTRUMENTATION ENGINEERING MASTER COURSE\n\n\n\nCHAPTER 2 · MODULE 01\n\nPhysical Quantities, Units, Geometry, and Engineering Conversions\n\nMass • Weight • Density • Metric Prefixes • Area • Volume • Dimensional Analysis\n\n\n\nMODULE PURPOSE\n\nEstablish the physics and measurement foundation required for instrumentation engineering by developing clear understanding of mass, force, acceleration, density, geometry, metric prefixes, dimensional analysis, and reliable unit conversion.\n\n\n\n\n\n\n1. Wh

In [3]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core import VectorStoreIndex
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter

from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb

 
# -------------------------------------------------
# 1. Load documents
# -------------------------------------------------

documents = SimpleDirectoryReader(
    input_dir="./data"
).load_data()

print(f"Loaded {len(documents)} documents")

Loaded 1 documents


# 2. Connect to local Ollama models

In [4]:
llm = Ollama(
    model="qwen3.5:9b",
    request_timeout=300.0,
    context_window=8192,
)

embed_model = OllamaEmbedding(
    model_name="nomic-embed-text",
    base_url="http://localhost:11434",
)

# 3. Create persistent Chroma database


In [5]:
chroma_client = chromadb.PersistentClient(
    path="./chroma_db"
)

chroma_collection = (
    chroma_client.get_or_create_collection(
        name="industrial_documents"
    )
)

vector_store = ChromaVectorStore(
    chroma_collection=chroma_collection
)

# 4. Split, embed, and store documents


In [6]:
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(
            chunk_size=512,
            chunk_overlap=50,
        ),
        embed_model,
    ],
    vector_store=vector_store,
)

nodes = pipeline.run(
    documents=documents,
    show_progress=True,
)

print(f"Created and stored {len(nodes)} nodes")

Applying transformations:   0%|          | 0/2 [00:00<?, ?it/s]

2026-08-03 21:02:59,841 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-08-03 21:03:00,016 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


Created and stored 11 nodes


# 5. Create the searchable index


In [7]:
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    embed_model=embed_model,
)

# 6. Create the query engine


In [9]:
query_engine = index.as_query_engine(
    llm=llm,
    similarity_top_k=4,
    response_mode="compact",
)


# 7. Ask a question

In [11]:
response = query_engine.query(
    "Explain . What is weight."
)

print(response)

2026-08-03 21:29:53,815 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-08-03 21:31:55,906 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Weight is defined as the gravitational force acting on a mass. Unlike mass, which represents an intrinsic property and resistance to acceleration that remains constant regardless of location such as Earth, the Moon, or orbit, weight depends entirely on the local gravitational field. It can be calculated using the formula $W = mg$, where it relates mass ($m$) multiplied by gravity ($g$). In terms of measurement units within the SI system, this force is expressed in newtons and is symbolized as W or Fg.
